# Introduction

This notebook is suppose to walk you through some of the practical details for the LMU Data Challenge "The Execution Edge".

The notebook focuses on two main aspects of the challenge: the submission format and the evaluation function. 

We start by loading the data...


In [4]:
#load basic libraries
import polars as pl  # It is advised to use polars, as the detasets can be quite memory-intensive
import numpy as np
import re

%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:

# Query data
# We load the BTCUSDT pair
data_direct = 'BTCUSDT'
X_train = pl.read_parquet(f"{data_direct}/X_train.parquet")
y_train = pl.read_parquet(f"{data_direct}/y_train.parquet")

# load the volume to fill info from the text file
with open(f"{data_direct}/vol_to_fill.txt", "r") as file:
    content = file.read().strip()

match = re.search(r"The volume to fill is: ([\d.]+)", content)
if match:
    volume_to_fill  = float(match.group(1))
    print(f"Extracted number: {volume_to_fill}")
else:
    print("No number found in the file.")


Extracted number: 4.0


# Submission example format. 

The submission format should be a dataframe with 3 columns: 'anonymized_id', 'time_in_hour', 'position'. 

Please make sure that the sum of the positions is "volume_to_fill". 

In [6]:
# This cell will show a Submission example

# Note: some seconds might be missing, also in the train set.
# However, we you should ALWAYS submit a completed position of length 60 for each anonymized_id.
# Here is an example of how to create a submission by filling the missing seconds with NaN

all_time_in_hour = y_train['time_in_hour'].unique().sort()
unique_ids = y_train['anonymized_id'].unique().sort()
templated_df = unique_ids.to_frame().join(
        all_time_in_hour.to_frame(),
        how="cross"
    )
# Fill the missing seconds with NaN
y_helper =templated_df.join(
        y_train,
        on=['anonymized_id', 'time_in_hour'],
        how="left"
    )

# as a simple example, we will take positions uniformly over the last 60 seconds
y_example_ask = y_helper.select(
    "anonymized_id",
    "time_in_hour"
).with_columns(
    position=pl.lit(volume_to_fill/60).alias("position")
)

display(y_example_ask.head()) # 3 columns: anonymized_id, time_in_hour, position



anonymized_id,time_in_hour,position
u64,duration[ms],f64
10076153343292355,59m,0.066667
10076153343292355,59m 1s,0.066667
10076153343292355,59m 2s,0.066667
10076153343292355,59m 3s,0.066667
10076153343292355,59m 4s,0.066667


# Scoring metric: minimise implementation error.

Having analysied the submission format, we can pass to the scoring function. 

Fundamentally, the quantity you will need to minimise is:

$$\frac{\lvert \text{average\_price} - \text{close\_price} \rvert}{\text{close\_price}}$$

where:

$$\text{close\_price} := y\_last\_min.\text{select}("close").\text{drop\_nulls}().\text{to\_series}().\text{last}()$$

$$\text{average\_price} := \frac{\text{total\_value\_executed}}{\text{total\_volume\_executed}}$$

with the running sum:

$$\text{total\_value\_executed} := \text{total\_value\_executed} + (\text{position\_to\_fill} \times \text{price}).$$

In other words, the close_price represents the last traded price for each hour, while the average_price is the volume-weighted average price. For instance, purchasing 0.5 units of volume at a price of 10 and another 0.5 units at a price of 20 would result in an average price of 15.

## Filling penalty 

The actual scoring function incorporates a multiplicative penalty: $$ \min(100, \frac{\text{volume\_to\_fill}}{\text{total\_volume\_executed}}),$$ 

making the final function to minimize to be 

$$\frac{\lvert \text{average\_price} - \text{close\_price} \rvert}{\text{close\_price}} \times \min\left(100, \frac{\text{volume\_to\_fill}}{\text{total\_volume\_executed}}\right)$$

The above metric is then averaged over every anonymised_id, providing a mean implementation error. 

## Unfilled positions 


Since y_test is hidden, a position at second s might not be completely filled, even if the entire order book is "walked" (or due to data unavailability). In such cases, the unfilled portion of the position is carried forward to second s+1. Note that this allows your algorithm to handle unfilled positions recursively: if positions at time s not filled, then what should I do? 

## Writing your own scoring function 

We leave the implementation of the specific scoring function that best suits their workflow to the reader. We also believe this will be an excellent exercise to familiarize yourself with the data and the problem.

In helping you asses your scoring function, I will provide below some results on the data we have being using so far. In the .py file, you can find the function to simulate a "walk" of the book together with its tests. Adapting such function to your workflow should be, hopefully, straightforward.  

P.S. For my scoring function, I've adopted the convention that ask-side positions are represented by positive numbers, and bid-side positions by negative numbers. Furthermore, the final result is multiplied by 10,000 to express it in basis points (bps), which is a common business convention and does not impact the underlying problem.


In [7]:
# The spread is the difference between the ask and bid positions normalised by 
# the mid price (average of first level bid and ask).
# It is a very common benchamark for these kind of problems,  you should try to beat it consistently.

# It is usually expressed in basis points (bps), where 1 bps = 0.01%.

spread = 20000 * (X_train['ask_price_1'] - X_train['bid_price_1'])/ (X_train['ask_price_1'] + X_train['bid_price_1'])
print(f"Spread mean: {spread.mean()}")

Spread mean: 1.2063831214438363


In [8]:

y_check = y_helper.join(y_example_ask, on=["anonymized_id", "time_in_hour"], how="left")
# result = compute_implementation_error(y_check, volume_to_fill= volume_to_fill)
# The compute_implementation_error is not provided, but it should be easy to derive from simulate_walk_the_book function in the .py file

# This strategy gives a score of 1.512 bps for the ask side.
# It does not beat the spread, but it is a good starting point.

In [9]:
# In accordance with my scoring function convention, I implement 
# the bid position as a negative value.

y_example_bid = y_helper.select(
    "anonymized_id",
    "time_in_hour"
).with_columns(
    position=pl.lit(-volume_to_fill/60).alias("position")
)

y_check = y_helper.join(y_example_bid, on=["anonymized_id", "time_in_hour"], how="left")
#result = compute_implementation_error(y_check, volume_to_fill= volume_to_fill) #the compute_implementation_error is not provided.

# The scoring for the bid side is 1.5825. 


# One parameter model

In [10]:
# Lets check a simpler model where we fill only positions close to the end of the hour.
# Getting closer to the end of the hour will bring the ask (or bid) price closer to the close price,
# but we might not fill the entire volume.

voc_results = {}
for x in range(1,60):
    time_bound = pl.duration(minutes=59, seconds= x)
    y_example = y_helper.select(
        "anonymized_id",
        "time_in_hour"
    ).with_columns(
        position=pl.when(pl.col("time_in_hour") < time_bound)
        .then(pl.lit(0 / 60 ))
        .otherwise(pl.lit(volume_to_fill / (60-x)))
        .alias("position")
    )

    y_check = y_helper.join(y_example, on=["anonymized_id", "time_in_hour"], how="left")
    #result = compute_implementation_error(y_check, volume_to_fill= volume_to_fill)
    #voc_results[x] = result



In [11]:
sorted_voc = sorted(voc_results.items(), key=lambda x: x[1])
print("Best time bound:", sorted_voc[0][0], "with score:", sorted_voc[0][1])
# we obtained 1.2186 bps with a time bound of 46, quite a decent improvement!


IndexError: list index out of range